# QNBAnalytics ML - Unified Credit Scoring Pipeline

**Bank of Baku Credit Scoring System - Training & Scoring**

Version: 0.3.2

This notebook combines both training and scoring workflows:
- **Part 1:** Train 3-layer hierarchical model
- **Part 2:** Score new applicants with policy adjustments

---

## Configuration & Setup

In [ ]:
# Import libraries
from QNBAnalytics_ML import data
from QNBAnalytics_ML import skills_api
from QNBAnalytics_ML.feature_importances import plot_feature_importances
from QNBAnalytics_ML.param_grid_best import grids as param_grid_best
import pandas as pd
import numpy as np
import pickle
import datetime

print("Libraries imported successfully!")

In [ ]:
# Configuration
version = "training"
start_time = datetime.datetime.now()

# Data columns
index_col = "MUQAVILE"
target_col = "TARGET"

# Columns to drop (data quality/leakage)
cols_to_drop = [
    "CC_O_4M6MWPS_EVER", "ALL_O_4M6MWPS_EVER_O", "ALL_O_7M12MWPS_EVER",
    "ALL_O_7M12MWPS_EVER_O", "ALL_OSMLMT_CWPS1_6_EVER", "CC_O_13M24MWPS_EVER",
    "CC_O_13M24MWPS_EVER_O", "CC_O_7M12MWPS_EVER", "CC_O_7M12MWPS_EVER_O",
    "CCOL_O_4M6MWPS_EVER", "CL_O_13M24MWPS_EVER_O", "HL_EVERWPS_EVER",
    "HL_O_13M24MWPS_365DP", "HL_O_3MWPS_365DP", "HL_O_4M6MWPS_EVER_O",
    "HL_O_7M12MWPS_EVER_O", "HL_O_EVERWPS_365DP", "OL_O_3MWPS_183D365D",
    "OL_O_3MWPS_365DP", "OL_O_4M6MWPS_183D365D", "OL_O_4M6MWPS_91D182D",
    "OL_O_7M12MWPS_365DP", "OL_O_EVERWPS_183D365D", "OL_O_EVERWPS_365DP",
    "OL_O_EVERWPS_90D", "OL_O_EVERWPS_91D182D", "OL_OLMTUTL_CWPS0_90D_O", "BGN"
]

# Credit score transformation parameters
ref = 200                      # Reference score
odds_at_ref_segmentation = 50  # For Layer 2 splits
odds_at_ref_scoring = 100      # For final scoring
points_to_double = 20          # Points to double odds

# Segmentation thresholds
good_score_threshold = 180
not_good_score_threshold = 200

# Policy adjustment
policy_constant = 250
policy_multiplier = 0.95

# Model selection for Layer 2
not_good_selected_model = 'Logistic Regression'
good_selected_model = 'LGBM'

print(f"Configuration loaded for version: {version}")

---
# PART 1: MODEL TRAINING
---

## 1. Load Training Data

In [ ]:
# Database connection
database_username = pd.read_table('Data/user', header=None)[0][0]
database_password = pd.read_table('Data/pass', header=None)[0][0]
engine = data.connect_to_sql(database_username, database_password)

# Load training data
with open(f'Data/train_data_sql_{version}.txt') as f:
    train_data_sql = f.read()
train = data.data_load(engine, sql=train_data_sql)

# Prepare training data
train = train.set_index(index_col).rename(columns={target_col: "TARGET"})
y_train = train["TARGET"]
x_train = train.drop(columns=["TARGET"] + cols_to_drop, errors='ignore')

print(f"Training data: {x_train.shape[0]} samples, {x_train.shape[1]} features")

# Load test data
engine = data.connect_to_sql(database_username, database_password)
with open(f'Data/test_data_sql_{version}.txt') as f:
    test_data_sql = f.read()
test = data.data_load(engine, sql=test_data_sql)

# Prepare test data
test = test.set_index(index_col).rename(columns={target_col: "TARGET"})
y_test = test["TARGET"]
x_test = test.drop(columns=["TARGET"] + cols_to_drop, errors='ignore')

print(f"Test data: {x_test.shape[0]} samples, {x_test.shape[1]} features")

## 2. Layer 1: Base Model Training

In [ ]:
# Layer 1 pipeline settings
apply_data_explore = True
apply_null_elimination = True
apply_constant_elimination = True
apply_low_gini_elimination = True
apply_correlated_feature_elimination = True
apply_binning = True
apply_scaling = False
apply_null_imputation = False
apply_categorical_encoding = False
apply_LR = True

In [ ]:
# Define Base Pipeline
class Pipeline_Base:
    def train(self, x_train, y_train, x_test=[], y_test=[]):
        if (len(x_test) > 0) & (len(y_test) > 0):
            x_val = x_test.copy()
            y_val = y_test.copy()
        
        self.pipeline = skills_api.ClassifierPipeline(x_train, y_train, x_val, y_val)
        
        if apply_data_explore:
            self.pipeline.data_explore()
        
        if apply_null_elimination:
            self.pipeline.feature_elimination(eliminator='drop_null_features', params={'threshold': 0.99})
        
        if apply_constant_elimination:
            self.pipeline.feature_elimination(eliminator='drop_constant_features', params={'missing_values':'ignore'})
        
        if apply_low_gini_elimination:
            self.pipeline.feature_elimination(eliminator='drop_low_gini_features', params={'threshold':0.05, 'missing_values':'ignore'})
        
        if apply_correlated_feature_elimination:
            self.pipeline.feature_elimination(eliminator='correlated_lower_gini_feature_elimination', params={'missing_values':'ignore'})
        
        if apply_binning:
            self.pipeline.binning(use_existing='binning.pkl')
        
        if apply_scaling:
            self.pipeline.scaling()
        
        if apply_null_imputation:
            self.pipeline.null_imputation()
        
        if apply_categorical_encoding:
            self.pipeline.encode_categoricals()
        
        if apply_LR:
            self.pipeline.LogisticRegression(param_opt=True, param_opt_method="RandomizedSearchCV", 
                                             train_size=x_train.shape[0], param_grid=param_grid_best['logistic_regression_base'])
    
    def apply(self, x_test, y_test=[]):
        predicts = self.pipeline.test(test=x_test, y_test=y_test)
        return predicts

In [ ]:
# Train Base Model
print("="*60)
print("LAYER 1: Training Base Model")
print("="*60)

pipeline_base = Pipeline_Base()
pipeline_base.train(x_train=x_train, y_train=y_train, x_test=x_test, y_test=y_test)

# Save model
model_path = f'Models/base_model_{version}.pkl'
with open(model_path, 'wb') as out:
    pickle.dump(pipeline_base.pipeline, out, pickle.HIGHEST_PROTOCOL)
print(f"\nModel saved: {model_path}")

## 3. Data Segmentation for Layer 2

In [ ]:
# Calculate probability thresholds (using odds_at_ref=50 for segmentation)
good_threshold = 1 / (np.exp((good_score_threshold - ref) / points_to_double * np.log(2) + np.log(odds_at_ref_segmentation)) + 1)
not_good_threshold = 1 / (np.exp((not_good_score_threshold - ref) / points_to_double * np.log(2) + np.log(odds_at_ref_segmentation)) + 1)

print(f"Good threshold (score {good_score_threshold}): {good_threshold:.6f}")
print(f"Not-Good threshold (score {not_good_score_threshold}): {not_good_threshold:.6f}")

# Get base predictions
train_preds = pipeline_base.apply(x_train, y_test=y_train)
base_test_proba_dict = train_preds
train_proba = train_preds["".join(key for key in train_preds if "Logistic Regression" in key)][0]

# Split training data
not_good_indices = train_proba >= not_good_threshold
good_indices = train_proba < good_threshold

train_not_good = train[not_good_indices]
train_good = train[good_indices]

x_train_not_good = train_not_good.drop(columns=["TARGET"] + cols_to_drop, errors='ignore')
y_train_not_good = train_not_good["TARGET"]
x_train_good = train_good.drop(columns=["TARGET"] + cols_to_drop, errors='ignore')
y_train_good = train_good["TARGET"]

print(f"\nNot-Good segment: {len(x_train_not_good)} samples")
print(f"Good segment: {len(x_train_good)} samples")

# Split test data
test_preds = pipeline_base.apply(x_test=x_test, y_test=y_test)
test_proba = test_preds["".join(key for key in test_preds if "Logistic Regression" in key)][0]

not_good_indices_test = test_proba >= not_good_threshold
good_indices_test = test_proba < good_threshold

test_not_good = test[not_good_indices_test]
test_good = test[good_indices_test]

x_test_not_good = test_not_good.drop(columns=["TARGET"] + cols_to_drop, errors='ignore')
y_test_not_good = test_not_good["TARGET"]
x_test_good = test_good.drop(columns=["TARGET"] + cols_to_drop, errors='ignore')
y_test_good = test_good["TARGET"]

## 4. Layer 2: Segment Model Training

In [ ]:
# Layer 2 pipeline settings
apply_data_explore = True
apply_null_elimination = True
apply_constant_elimination = True
apply_low_gini_elimination = True
apply_correlated_feature_elimination = True
apply_binning = False
apply_scaling = True
apply_null_imputation = True
apply_categorical_encoding = True

apply_LR = True
apply_RF = True
apply_XGB = True
apply_LGBM = True

In [ ]:
# Define Segment Pipelines
class Pipeline_Segment:
    def train(self, x_train, y_train, x_test=[], y_test=[], segment_type='not_good'):
        if (len(x_test) > 0) & (len(y_test) > 0):
            x_val = x_test.copy()
            y_val = y_test.copy()
        
        self.pipeline = skills_api.ClassifierPipeline(x_train, y_train, x_val, y_val)
        
        if apply_data_explore:
            self.pipeline.data_explore()
        if apply_null_elimination:
            self.pipeline.feature_elimination(eliminator='drop_null_features', params={'threshold': 0.99})
        if apply_constant_elimination:
            self.pipeline.feature_elimination(eliminator='drop_constant_features', params={'missing_values':'ignore'})
        if apply_low_gini_elimination:
            self.pipeline.feature_elimination(eliminator='drop_low_gini_features', params={'threshold':0.05, 'missing_values':'ignore'})
        if apply_correlated_feature_elimination:
            self.pipeline.feature_elimination(eliminator='correlated_lower_gini_feature_elimination', params={'missing_values':'ignore'})
        if apply_binning:
            self.pipeline.binning()
        if apply_scaling:
            self.pipeline.scaling()
        if apply_null_imputation:
            self.pipeline.null_imputation()
        if apply_categorical_encoding:
            self.pipeline.encode_categoricals()
        
        # Get param grids
        lr_grid = param_grid_best[f'logistic_regression_{segment_type}']
        rf_grid = param_grid_best[f'random_forest_{segment_type}']
        xgb_grid = param_grid_best[f'xgboost_{segment_type}']
        lgbm_grid = param_grid_best[f'lightgbm_{segment_type}']
        
        if apply_LR:
            self.pipeline.LogisticRegression(param_opt=True, param_opt_method="RandomizedSearchCV", 
                                             train_size=x_train.shape[0], param_grid=lr_grid)
        if apply_RF:
            self.pipeline.RandomForest(param_opt=True, param_opt_method="RandomizedSearchCV", 
                                       train_size=x_train.shape[0], param_grid=rf_grid)
        if apply_XGB:
            self.pipeline.XGBoost(param_opt=True, param_opt_method="RandomizedSearchCV", 
                                  train_size=x_train.shape[0], param_grid=xgb_grid)
        if apply_LGBM:
            self.pipeline.LGBM(param_opt=True, param_opt_method="RandomizedSearchCV", 
                               train_size=x_train.shape[0], param_grid=lgbm_grid)
    
    def apply(self, x_test, y_test=[]):
        predicts = self.pipeline.test(test=x_test, y_test=y_test)
        return predicts

In [ ]:
# Train Not-Good Segment
print("="*60)
print("LAYER 2: Training Not-Good Segment")
print("="*60)

pipeline_not_good = Pipeline_Segment()
pipeline_not_good.train(x_train=x_train_not_good, y_train=y_train_not_good, 
                        x_test=x_test_not_good, y_test=y_test_not_good, segment_type='not_good')

# Save model
with open(f'Models/not_good_model_{version}.pkl', 'wb') as out:
    pickle.dump(pipeline_not_good.pipeline, out, pickle.HIGHEST_PROTOCOL)

# Train Good Segment
print("\n" + "="*60)
print("LAYER 2: Training Good Segment")
print("="*60)

pipeline_good = Pipeline_Segment()
pipeline_good.train(x_train=x_train_good, y_train=y_train_good, 
                    x_test=x_test_good, y_test=y_test_good, segment_type='good')

# Save model
with open(f'Models/good_model_{version}.pkl', 'wb') as out:
    pickle.dump(pipeline_good.pipeline, out, pickle.HIGHEST_PROTOCOL)

## 5. Layer 3: Meta Model Training

In [ ]:
# Generate meta features for training
base_train_proba = pipeline_base.apply(x_train, y_train)["".join(key for key in base_test_proba_dict if "Logistic Regression" in key)][0]
not_good_train_proba_dict = pipeline_not_good.apply(x_train, y_train)
not_good_train_proba = not_good_train_proba_dict["".join(key for key in not_good_train_proba_dict if f"{not_good_selected_model}" in key)][0]
good_train_proba_dict = pipeline_good.apply(x_train, y_train)
good_train_proba = good_train_proba_dict["".join(key for key in good_train_proba_dict if f"{good_selected_model}" in key)][0]

x_train_meta = pd.DataFrame({
    'Base': base_train_proba,
    'Good': good_train_proba,
    'Not_Good': not_good_train_proba
}, index=y_train.index)

# Generate meta features for test
base_test_proba = pipeline_base.apply(x_test, y_test)["".join(key for key in base_test_proba_dict if "Logistic Regression" in key)][0]
not_good_test_proba_dict = pipeline_not_good.apply(x_test, y_test)
not_good_test_proba = not_good_test_proba_dict["".join(key for key in not_good_test_proba_dict if f"{not_good_selected_model}" in key)][0]
good_test_proba_dict = pipeline_good.apply(x_test, y_test)
good_test_proba = good_test_proba_dict["".join(key for key in good_test_proba_dict if f"{good_selected_model}" in key)][0]

x_test_meta = pd.DataFrame({
    'Base': base_test_proba,
    'Good': good_test_proba,
    'Not_Good': not_good_test_proba
}, index=y_test.index)

print(f"Meta features created: {x_train_meta.shape}")

In [ ]:
# Layer 3 pipeline settings
apply_data_explore = True
apply_null_elimination = False
apply_constant_elimination = False
apply_correlated_feature_elimination = False
apply_low_gini_elimination = False
apply_binning = False
apply_scaling = False
apply_null_imputation = False
apply_categorical_encoding = False
apply_LR = True

In [ ]:
# Define Meta Pipeline
class Pipeline_Meta:
    def train(self, x_train, y_train, x_test=[], y_test=[]):
        if (len(x_test) > 0) & (len(y_test) > 0):
            x_val = x_test.copy()
            y_val = y_test.copy()
        
        self.pipeline = skills_api.ClassifierPipeline(x_train, y_train, x_val, y_val)
        
        if apply_data_explore:
            self.pipeline.data_explore()
        
        if apply_LR:
            self.pipeline.LogisticRegression(param_opt=True, param_opt_method="RandomizedSearchCV", 
                                             train_size=x_train.shape[0], param_grid=param_grid_best['logistic_regression_meta'])
    
    def apply(self, x_test, y_test=[]):
        predicts = self.pipeline.test(test=x_test, y_test=y_test)
        return predicts

In [ ]:
# Train Meta Model
print("="*60)
print("LAYER 3: Training Meta Model")
print("="*60)

pipeline_meta = Pipeline_Meta()
pipeline_meta.train(x_train=x_train_meta, y_train=y_train, x_test=x_test_meta, y_test=y_test)

# Save model
with open(f'Models/meta_model_{version}.pkl', 'wb') as out:
    pickle.dump(pipeline_meta.pipeline, out, pickle.HIGHEST_PROTOCOL)

print("\nAll models trained and saved!")
print(f"Training time: {datetime.datetime.now() - start_time}")

## 6. Calculate Final Training Scores

In [ ]:
# Get final predictions
meta_test_proba_dict = pipeline_meta.apply(x_test_meta, y_test_meta=y_test)
meta_test_proba = meta_test_proba_dict["".join(key for key in meta_test_proba_dict if "Logistic Regression" in key)][0]

# Apply to all data
all_x = pd.concat([x_train, x_test], axis=0)
all_y = pd.concat([y_train, y_test], axis=0)

base_all_proba = pipeline_base.apply(all_x, all_y)["".join(key for key in base_test_proba_dict if "Logistic Regression" in key)][0]
not_good_all_proba = pipeline_not_good.apply(all_x, all_y)["".join(key for key in not_good_train_proba_dict if f"{not_good_selected_model}" in key)][0]
good_all_proba = pipeline_good.apply(all_x, all_y)["".join(key for key in good_train_proba_dict if f"{good_selected_model}" in key)][0]

x_all_meta = pd.DataFrame({
    'Base': base_all_proba,
    'Good': good_all_proba,
    'Not_Good': not_good_all_proba
}, index=all_y.index)

meta_all_proba = pipeline_meta.apply(x_all_meta, y_all_meta=all_y)["".join(key for key in meta_test_proba_dict if "Logistic Regression" in key)][0]

# Convert to scores (using odds_at_ref=100 for final scoring)
default_rate = np.where(meta_all_proba == 0, 0.00001, meta_all_proba)
odds = (1/default_rate) - 1
meta_all_score = ((np.log(odds) - np.log(odds_at_ref_scoring)) / np.log(2)) * points_to_double + ref

# Save results
results = pd.DataFrame({
    'PROBA': meta_all_proba,
    'SCORE': meta_all_score,
    'TARGET': all_y
}, index=all_y.index)

results.to_excel(f'Output/TRAINING_SCORES_{version}.xlsx')
print(f"\nFinal scores saved to: Output/TRAINING_SCORES_{version}.xlsx")
print(f"Score range: {meta_all_score.min():.1f} - {meta_all_score.max():.1f}")

---
# PART 2: SCORING NEW APPLICANTS
---

## 7. Load Trained Models

In [ ]:
# Load all trained models
print("Loading trained models...")

with open(f'Models/base_model_{version}.pkl', 'rb') as f:
    pipeline_base_loaded = pickle.load(f)

with open(f'Models/good_model_{version}.pkl', 'rb') as f:
    pipeline_good_loaded = pickle.load(f)

with open(f'Models/not_good_model_{version}.pkl', 'rb') as f:
    pipeline_not_good_loaded = pickle.load(f)

with open(f'Models/meta_model_{version}.pkl', 'rb') as f:
    pipeline_meta_loaded = pickle.load(f)

print("All models loaded successfully!")

## 8. Load Scoring Data

In [ ]:
# Load new applicants for scoring
engine = data.connect_to_sql(database_username, database_password)
with open('Data/test_data_sql_scoring_policy_adjustment.txt') as f:
    scoring_sql = f.read()

scoring_data = data.data_load(engine, sql=scoring_sql)

# For scoring, index column might be 'ID' instead of 'MUQAVILE'
scoring_index_col = 'ID' if 'ID' in scoring_data.columns else index_col

scoring_data = scoring_data.set_index(scoring_index_col).rename(columns={target_col: "TARGET"})
y_scoring = scoring_data["TARGET"] if "TARGET" in scoring_data.columns else None
x_scoring = scoring_data.drop(columns=["TARGET"] + cols_to_drop, errors='ignore')

print(f"Scoring data: {x_scoring.shape[0]} applicants, {x_scoring.shape[1]} features")

## 9. Apply Models (Layer 1 → 2 → 3)

In [ ]:
# Layer 1: Base model
print("Applying Layer 1 (Base)...")
base_scoring_proba_dict = pipeline_base_loaded.test(test=x_scoring, y_test=y_scoring if y_scoring is not None else [])
base_scoring_proba = base_scoring_proba_dict["".join(key for key in base_scoring_proba_dict if "Logistic Regression" in key)][0]

# Layer 2: Segment models
print("Applying Layer 2 (Segments)...")
not_good_scoring_proba_dict = pipeline_not_good_loaded.test(test=x_scoring, y_test=y_scoring if y_scoring is not None else [])
not_good_scoring_proba = not_good_scoring_proba_dict["".join(key for key in not_good_scoring_proba_dict if f"{not_good_selected_model}" in key)][0]

good_scoring_proba_dict = pipeline_good_loaded.test(test=x_scoring, y_test=y_scoring if y_scoring is not None else [])
good_scoring_proba = good_scoring_proba_dict["".join(key for key in good_scoring_proba_dict if f"{good_selected_model}" in key)][0]

# Layer 3: Meta model
print("Applying Layer 3 (Meta)...")
x_scoring_meta = pd.DataFrame({
    'Base': base_scoring_proba,
    'Good': good_scoring_proba,
    'Not_Good': not_good_scoring_proba
}, index=x_scoring.index)

meta_scoring_proba_dict = pipeline_meta_loaded.test(test=x_scoring_meta, y_test=y_scoring if y_scoring is not None else [])
meta_scoring_proba = meta_scoring_proba_dict["".join(key for key in meta_scoring_proba_dict if "Logistic Regression" in key)][0]

print("All layers applied successfully!")

## 10. Calculate Credit Scores & Apply Policy

In [ ]:
# Convert probability to raw score
default_rate = np.where(meta_scoring_proba == 0, 0.00001, meta_scoring_proba)
odds = (1/default_rate) - 1
raw_score = ((np.log(odds) - np.log(odds_at_ref_scoring)) / np.log(2)) * points_to_double + ref

# Apply policy adjustments
final_score = np.minimum(
    raw_score,
    np.minimum(
        policy_constant,  # Cap at 250
        raw_score * policy_multiplier  # 5% discount
    )
)

print(f"Raw score range: {raw_score.min():.1f} - {raw_score.max():.1f}")
print(f"Final score range: {final_score.min():.1f} - {final_score.max():.1f}")
print(f"Average adjustment: {(raw_score - final_score).mean():.2f} points")

## 11. Save Scoring Results

In [ ]:
# Create results dataframe
scoring_results = pd.DataFrame({
    'PROBA': meta_scoring_proba,
    'RAW_SCORE': raw_score,
    'FINAL_SCORE': final_score
}, index=x_scoring.index)

if y_scoring is not None:
    scoring_results['TARGET'] = y_scoring

# Save to Excel
output_path = f'Output/SCORING_RESULTS_{version}.xlsx'
scoring_results.to_excel(output_path)

print(f"\nScoring results saved to: {output_path}")
print(f"Total applicants scored: {len(scoring_results)}")

# Display summary statistics
print("\n" + "="*60)
print("SCORING SUMMARY")
print("="*60)
print(scoring_results.describe())

---
## Pipeline Complete!

**Training outputs:**
- `Models/base_model_training.pkl`
- `Models/good_model_training.pkl`
- `Models/not_good_model_training.pkl`
- `Models/meta_model_training.pkl`
- `Output/TRAINING_SCORES_training.xlsx`

**Scoring outputs:**
- `Output/SCORING_RESULTS_training.xlsx`

---